# Adding semantic tags to an sqlite table

This notebook adds semantic tags to adverbials in the Estonian Reference corpus. The semantic types are added to two sqlite database tables *spatial_obl* and *advmod* into a new column called *ekilex_tag*.

In [1]:
#imports
import sqlite3
import os
import re

In [2]:
def column_exists(cursor, table, column):
    cursor.execute(f"PRAGMA table_info({table})")
    return any(row[1] == column for row in cursor.fetchall())

In [ ]:
# database file path
DATABASE = "../drive_data/v33_koondkorpus_transaktsioonid_v04_2.db"

NER_TIMEX_DB = "../drive_data/v33_ner_timex_20250922-094340.db"

DB2 = "../drive_data/v33_koondkorpus_transaktsioonid_v05.db"


In [10]:
# connecting with database
conn_db = sqlite3.connect(DATABASE)
cursor_db = conn_db.cursor()

### Save word and semantic type to dict

In [4]:
def word_semtype_fun(directory_str):
    word_semtype = [] # tuples of semtype and word, i.e. (amount, aegsamini)
    
    directory = os.fsencode(directory_str)
    
    for file in os.listdir(directory):
        file_name = os.fsdecode(file)
        filepath = directory_str + '/' + file_name
        semtype = re.findall(r"^(?:adv_)?(.+?)\.[^.]+$", file_name)
        with open(filepath, "r", encoding="utf-8") as f:
            words = f.read().splitlines()
            for word in words:
                word_semtype.append((semtype[0], word))  
    return word_semtype

In [5]:
directory_obl =  "../base_data/v05_wordlists_obl"
word_semtype_obl = word_semtype_fun(directory_obl)

In [6]:
directory_advmod =  "../base_data/v05_wordlists_advmod"
word_semtype_adv = word_semtype_fun(directory_advmod)

In [7]:
word_semtype_adv

[('amount', 'aegsamini'),
 ('amount', 'ainumalt'),
 ('amount', 'ammendavalt'),
 ('amount', 'arvamata'),
 ('amount', 'aupoolest'),
 ('amount', 'aupärast'),
 ('amount', 'enam-vähem'),
 ('amount', 'esimeseks'),
 ('amount', 'esiteks'),
 ('amount', 'etemini'),
 ('amount', 'fantastiliselt'),
 ('amount', 'forsseeritult'),
 ('amount', 'haaravalt'),
 ('amount', 'halvemini'),
 ('amount', 'halvimini'),
 ('amount', 'hambuni'),
 ('amount', 'harvanähtavalt'),
 ('amount', 'hiiglamoodi'),
 ('amount', 'hinge põhjani'),
 ('amount', 'hingepõhjani'),
 ('amount', 'hinge põhjast'),
 ('amount', 'hingepõhjast'),
 ('amount', 'hirmpalju'),
 ('amount', 'hulga'),
 ('amount', 'hulganiselt'),
 ('amount', 'hulgim'),
 ('amount', 'hullu'),
 ('amount', 'hullult'),
 ('amount', 'hullumoodi'),
 ('amount', 'hullupööra'),
 ('amount', 'hullusti'),
 ('amount', 'häbemata'),
 ('amount', 'ilmama'),
 ('amount', 'imeharva'),
 ('amount', 'imehästi'),
 ('amount', 'imevähe'),
 ('amount', 'issanda'),
 ('amount', 'jalaga segada'),
 ('a

### Add semantic types to lemmas in the database

In [8]:
def semtype_to_db(table_name, semtypes, cursor, conn):
    # Step 1: add new column to database table
    if not column_exists(cursor, table_name, "ekilex_tag"):
        cursor.execute("ALTER TABLE " + table_name + " ADD COLUMN ekilex_tag TEXT")

    # Step 2: add word + semantic tag to temporary table
    cursor.execute("CREATE TEMP TABLE IF NOT EXISTS temp_updates (lemma TEXT PRIMARY KEY, ekilex_tag TEXT)")
    cursor.executemany("INSERT INTO temp_updates (ekilex_tag, lemma) VALUES (?, ?)", semtypes)

    # Step 3: Add temporary table info to database table
    #ps, pronouns are excluded for spatial obliques
    cursor.execute(f"""
        UPDATE {table_name}
        SET ekilex_tag = (SELECT ekilex_tag FROM temp_updates WHERE temp_updates.lemma = {table_name}.lemma)
        WHERE pos != 'P' AND EXISTS (SELECT 1 FROM temp_updates WHERE temp_updates.lemma = {table_name}.lemma)
    """)

    conn.commit()
    conn.close()

In [9]:
semtype_to_db('spatial_obl', word_semtype_obl, cursor_db, conn_db)

In [11]:
semtype_to_db('advmod', word_semtype_adv, cursor_db, conn_db)

In [12]:
conn_db.close()

## juurde lisatud ner&timex jaoks

## Add NER & TIMEX

In [13]:
import pandas as pd

In [14]:
# connecting with database
conn = sqlite3.connect(NER_TIMEX_DB)
cursor = conn.cursor()

In [15]:
# get ner and timex tables
query = f"SELECT * FROM ner"
ner = pd.read_sql(query, conn)

query = f"SELECT * FROM timex"
timex = pd.read_sql(query, conn)

In [16]:
conn.close()

In [21]:
# add ner and timex to the other db file

# connecting with database
conn = sqlite3.connect(DATABASE)
cursor = conn.cursor()

In [18]:
ner.to_sql("ner", conn, if_exists="replace", index=False)
timex.to_sql("timex", conn, if_exists="replace", index=False)

2179320

## Update spatial_obl table

In [19]:
if not column_exists(cursor, "spatial_obl", "ner_tag"):
    cursor.execute("ALTER TABLE " + "spatial_obl" +  " ADD COLUMN ner_tag TEXT")

In [20]:
# Update `table1.x` with values from `table2.x`
cursor.execute("""
        UPDATE spatial_obl
        SET ner_tag = ner.ner_tag
        FROM ner
        WHERE spatial_obl.sentence_id = ner.sentence_id and spatial_obl.row_loc=ner.loc;
    """)

conn.commit()
conn.close()

In [22]:
if not column_exists(cursor, "spatial_obl", "timex_tag"):
    cursor.execute("ALTER TABLE " + "spatial_obl" +  " ADD COLUMN timex_tag TEXT")

In [23]:
# Update `table1.x` with values from `table2.x`
cursor.execute("""
        UPDATE spatial_obl
        SET timex_tag = timex.timex_type
        FROM timex
        WHERE spatial_obl.sentence_id = timex.sentence_id and spatial_obl.row_loc=timex.loc;
    """)

conn.commit()
conn.close()

In [24]:
# kontroll
# connecting with database
conn = sqlite3.connect(DB2)
cursor = conn.cursor()

In [25]:
query = f"SELECT * FROM spatial_obl where ner_tag is not null limit 10"

res = pd.read_sql(query, conn)
res

,id,head_id,form,lemma,feats,pos,verb,verb_compound,sentence_id,row_loc,sentence,ekilex_tag,ner_tag,timex_tag
0,444,282,Pariisis,Pariis,"in,prop,sg",S,külastama,,160,5,"Vahepeal , maikuus külastasin Pariisis veel Ol...",location,LOC,None
1,773,481,linnal,linn,"ad,com,sg",S,tulema,,259,30,Aivar Mäe arvab aasta tegu ehk Pärnu Kontserdi...,None,LOC,None
2,1017,613,Eestis,Eesti,"in,prop,sg",S,väärima,,333,1,Eestis väärib oma tiitlit Erki Nool .,location,LOC,None
3,1586,953,Robbie'le,Robbie,"all,prop,sg",S,meeldima,,525,3,Kuuldavasti meeldib Robbie'le ka kihla vedada ...,None,PER,None
4,1743,1026,Tiinale,Tiina,"all,prop,sg",S,loovutama,,578,4,Lahutades loovutas viimane Tiinale aktsiaid ni...,None,PER,None
5,1807,1055,Tallinnas,Tallinn,"in,prop,sg",S,kandideerima,,597,5,1999. aastal kandideeris ta Tallinnas kohalike...,location,LOC,None
6,1838,1069,Neiveltil,Neivelt,"ad,prop,sg",S,soovitama,,605,4,"Tiina Mõis soovitas Neiveltil , keda ta ise on...",None,LOC,None
7,1866,1085,Otepäält,Otepää,"abl,prop,sg",S,kolima,,614,14,Kui Tiina vanem õde Maie koos olümpiavõitjast ...,location,LOC,None
8,1867,1085,Tallinnasse,Tallinn,"ill,prop,sg",S,kolima,,614,15,Kui Tiina vanem õde Maie koos olümpiavõitjast ...,location,LOC,None
9,1886,1090,koolis,kool,"com,in,sg",S,töötama,,617,7,"Jaak Uudmäe töötab Rocca al Mare koolis , mill...",None,ORG,None


In [26]:
query = f"SELECT * FROM spatial_obl where timex_tag is not null limit 10"

res = pd.read_sql(query, conn)
res

,id,head_id,form,lemma,feats,pos,verb,verb_compound,sentence_id,row_loc,sentence,ekilex_tag,ner_tag,timex_tag
0,1,2,lõpus,lõpp,"com,in,sg",S,toimuma,,3,3,Kaheksakümnendate aastate lõpus toimus Türi 1.,None,None,DATE
1,129,84,ajal,aeg,"ad,com,sg",S,loobuma,,54,3,Olen viimasel ajal juhutöödest loobunud ja kir...,None,None,DATE
2,155,98,hommikust,hommik,"com,el,sg",S,jooma,,63,18,"Ma ei saa aru inimestest , kes üldse napsu ei ...",None,None,TIME
3,205,142,hommikust,hommik,"com,el,sg",S,helistama,,84,14,"Mõned tüdrukud muutusid lausa tüütuks , uurisi...",None,None,TIME
4,221,157,aastal,aasta,"ad,com,sg",S,täitma,,92,2,"Sel aastal täitsime oma missiooni , mille eesm...",None,None,DATE
5,281,190,kevadel,kevad,"ad,com,sg",S,kavatsema,,110,13,Mul tekkis viis aastat tagasi tõsine huvi lenn...,time,None,DATE
6,346,231,oktoobris,oktoober,"com,in,sg",S,saama,,133,7,Priit : Tuttavaks saime 1998. aasta oktoobris ...,None,None,DATE
7,377,245,päeval,päev,"ad,com,sg",S,sattuma,,141,8,Aga Võitlevasse Sõnasse sattusin täiesti juhus...,None,None,DATE
8,416,271,aastal,aasta,"ad,com,sg",S,naasma,,153,11,"Olin omadega puntras , sest just enne seda , 1...",None,None,DATE
9,443,282,maikuus,maikuu,"com,in,sg",S,külastama,,160,3,"Vahepeal , maikuus külastasin Pariisis veel Ol...",None,None,DATE


In [27]:
conn.close()